In [2]:
!pip install langchain_google_genai
!pip install pyspark

In [3]:
import pandas as pd
import numpy as np
import os
import time
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
from langchain_google_genai import ChatGoogleGenerativeAI

# Machine Learning & Big Data Tools
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn.svm import LinearSVC
from pyspark.ml.classification import LogisticRegression as SparkLR, RandomForestClassifier as SparkRF, GBTClassifier, LinearSVC as SparkSVM

os.environ["GOOGLE_API_KEY"] = "PUT_YOUR_API_KEY"
# Initialize Gemini 2.5 Flash via LangChain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

# Download and Load Dataset
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(os.path.join(path, "creditcard.csv"))

Using Colab cache for faster access to the 'creditcardfraud' dataset.


In [ ]:
# 1. Scaling Amount and Time
scaler = RobustScaler()
df[['Amount', 'Time']] = scaler.fit_transform(df[['Amount', 'Time']])

# 2. Train/Test Split
X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# إنشاء جلسة Spark, يمكن تغيير الرقم من 4 الى اي رقم لتحديدد عدد Workers
spark = SparkSession.builder.master("local[4]").getOrCreate()

# تحويل بيانات Pandas إلى تنسيق Spark الموزع
train_spark = spark.createDataFrame(pd.concat([X_train, y_train], axis=1))
test_spark = spark.createDataFrame(pd.concat([X_test, y_test], axis=1))

# تجميع الميزات في Vector Assembler
feature_cols = X.columns.tolist()
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

train_final = assembler.transform(train_spark).select("features", "Class")
test_final = assembler.transform(test_spark).select("features", "Class")

In [ ]:
def run_experiment(name, sk_model, sp_class):
    # Sklearn
    t0 = time.time()
    sk_model.fit(X_train, y_train)
    sk_time = time.time() - t0
    sk_f1 = f1_score(y_test, sk_model.predict(X_test))

    t1 = time.time()
    sp_model = sp_class.fit(train_final)
    sp_time = time.time() - t1

    evaluator = MulticlassClassificationEvaluator(labelCol="Class", predictionCol="prediction", metricName="f1")
    sp_f1 = evaluator.evaluate(sp_model.transform(test_final))

    speedup = sk_time / sp_time if sp_time > 0 else 0

    print(f"\n[{name}] Results:")
    print(f"   - Sklearn: Time={sk_time:.2f}s, F1={sk_f1:.4f}")
    print(f"   - PySpark: Time={sp_time:.2f}s, F1={sp_f1:.4f}")
    print(f"   - Speedup: {speedup:.2f}x")
    return sp_model

lr_model  = run_experiment("Logistic Regression", LogisticRegression(max_iter=1000), SparkLR(labelCol="Class"))
svm_model = run_experiment("SVM", LinearSVC(max_iter=1000), SparkSVM(labelCol="Class"))
rf_model  = run_experiment("Random Forest", RandomForestClassifier(n_estimators=100), SparkRF(labelCol="Class"))
gbt_model = run_experiment("GBT (Gradient Boosting)", GradientBoostingClassifier(n_estimators=50), GBTClassifier(maxIter=50, labelCol="Class"))


[Logistic Regression] Results:
   - PySpark: Time=25.28s, F1=0.9991

[SVM] Results:
   - PySpark: Time=61.38s, F1=0.9993

[Random Forest] Results:
   - PySpark: Time=4.66s, F1=0.9994

[GBT (Gradient Boosting)] Results:
   - PySpark: Time=31.34s, F1=0.9994


In [ ]:
# الحصول على أهمية الميزات من نموذج Spark
importances = gbt_model.featureImportances.toArray()
feat_imp_series = pd.Series(importances, index=X.columns).sort_values(ascending=False)

#أعلى 3 ميزات لتفسيرها
top_3_features = feat_imp_series.head(3).index.tolist()
print(f"\nTop 3 Features for Reasoning: {top_3_features}")


Top 3 Features for Reasoning: ['V12', 'V26', 'V14']


In [ ]:
def generate_fraud_explanation(features, sample_data):
    prompt = f"""
    Context: You are a Financial Audit Expert.
    Incident: A credit card transaction was flagged as FRAUD by our PySpark system.
    Statistical Evidence: The most influential features were {features}.
    Transaction Data Summary: {sample_data}

    Task: Write a professional, concise explanation for a bank manager.
    Explain why these features (like V14 or V17) suggest a fraudulent pattern in this specific case.
    Note: your response using both english and arabic languages.
    """
    try:
        response = llm.invoke(prompt)
        return response.content
    except Exception as e:
        return f"LLM Connection Error: {e}"

# اختيار عينة احتيال حقيقية من بيانات الاختبار لإرسالها للتفسير
fraud_sample = X_test[y_test == 1].iloc[0].to_dict()

print("\n--- LLM Interpretive Report (Generated by Gemini) ---")
print(generate_fraud_explanation(top_3_features, fraud_sample))


--- LLM Interpretive Report (Generated by Gemini) ---
**To the Bank Manager:**

Our PySpark fraud detection system has flagged this credit card transaction as fraudulent. The primary indicators for this classification are the highly anomalous values observed in features V12, V14, and V26.

Specifically:
*   **V12 (-4.69) and V14 (-6.17):** These features, along with V17 (-6.54), exhibit extremely negative values. In our fraud models, such severe deviations from typical patterns in these specific features often signify highly unusual transaction characteristics, frequently associated with manipulative or unauthorized activity. They suggest a significant departure from legitimate transaction profiles.
*   **V26 (0.76):** The notably high positive value in V26 further contributes to the overall anomalous pattern, indicating another aspect of this transaction that falls outside normal operational bounds.

The combination of these extreme feature values strongly suggests a high-risk, poten